<a href="https://colab.research.google.com/github/alelopez9411-ux/torre-control-logistica/blob/main/notebooks/olist_torre_control.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery

PROYECTO = "olist-logistica"
client = bigquery.Client(project=PROYECTO)

print(list(client.list_datasets()))

In [3]:
import glob, os

for ruta in sorted(glob.glob("*.csv")):
    nombre = os.path.basename(ruta).replace("olist_", "").replace("_dataset.csv", "").replace(".csv", "")
    tabla = f"{PROYECTO}.olist.{nombre}"

    config = bigquery.LoadJobConfig(
        source_format=bigquery.SourceFormat.CSV,
        skip_leading_rows=1,
        autodetect=True,
        allow_quoted_newlines=True,
        write_disposition="WRITE_TRUNCATE",
    )

    with open(ruta, "rb") as f:
        job = client.load_table_from_file(f, tabla, job_config=config)
    job.result()

    print(f"{nombre:35s} -> {client.get_table(tabla).num_rows:>9,} filas")

customers                           ->    99,441 filas
geolocation                         -> 1,000,163 filas
order_items                         ->   112,650 filas
order_payments                      ->   103,886 filas
order_reviews                       ->    99,224 filas
orders                              ->    99,441 filas
products                            ->    32,951 filas
sellers                             ->     3,095 filas
product_category_name_translation   ->        71 filas


In [4]:
def sql(consulta):
    return client.query(consulta).to_dataframe()

# 1) Tipos de columna de la tabla de pedidos
for col in client.get_table(f"{PROYECTO}.olist.orders").schema:
    print(f"{col.name:35s} {col.field_type}")

print()

# 2) Pedidos por estado
sql("""
SELECT order_status, COUNT(*) AS pedidos
FROM `olist-logistica.olist.orders`
GROUP BY order_status
ORDER BY pedidos DESC
""")

order_id                            STRING
customer_id                         STRING
order_status                        STRING
order_purchase_timestamp            TIMESTAMP
order_approved_at                   TIMESTAMP
order_delivered_carrier_date        TIMESTAMP
order_delivered_customer_date       TIMESTAMP
order_estimated_delivery_date       TIMESTAMP



,order_status,pedidos
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2


In [5]:
base = """
WITH entregas AS (
  SELECT
    order_id,
    order_purchase_timestamp,
    TIMESTAMP_DIFF(order_delivered_customer_date, order_purchase_timestamp, HOUR) / 24 AS dias_real,
    TIMESTAMP_DIFF(order_estimated_delivery_date, order_purchase_timestamp, HOUR) / 24 AS dias_prometidos,
    IF(order_delivered_customer_date > order_estimated_delivery_date, 1, 0) AS tardio
  FROM `olist-logistica.olist.orders`
  WHERE order_status = 'delivered'
    AND order_delivered_customer_date IS NOT NULL
)
"""

kpi = sql(base + """
SELECT
  COUNT(*) AS entregados,
  ROUND(AVG(dias_real), 1) AS dias_real_promedio,
  ROUND(AVG(dias_prometidos), 1) AS dias_prometidos_promedio,
  ROUND(100 * AVG(tardio), 1) AS pct_tardios
FROM entregas
""")

mensual = sql(base + """
SELECT
  FORMAT_DATE('%Y-%m', DATE(order_purchase_timestamp)) AS mes,
  COUNT(*) AS entregados,
  ROUND(AVG(dias_real), 1) AS dias_real,
  ROUND(100 * AVG(tardio), 1) AS pct_tardios
FROM entregas
GROUP BY mes
ORDER BY mes
""")

display(kpi)
display(mensual)

,entregados,dias_real_promedio,dias_prometidos_promedio,pct_tardios
0,96470,12.5,23.7,8.1


,mes,entregados,dias_real,pct_tardios
0,2016-09,1,54.8,100.0
1,2016-10,265,19.6,1.1
2,2016-12,1,4.7,0.0
3,2017-01,750,12.6,3.1
4,2017-02,1653,13.1,3.2
5,2017-03,2546,12.9,5.6
6,2017-04,2303,14.9,7.9
7,2017-05,3545,11.3,3.6
8,2017-06,3135,12.0,3.9
9,2017-07,3872,11.6,3.4


In [6]:
base = """
WITH entregas AS (
  SELECT
    order_id,
    customer_id,
    order_purchase_timestamp,
    TIMESTAMP_DIFF(order_delivered_customer_date, order_purchase_timestamp, HOUR) / 24 AS dias_real,
    TIMESTAMP_DIFF(order_estimated_delivery_date, order_purchase_timestamp, HOUR) / 24 AS dias_prometidos,
    IF(order_delivered_customer_date > order_estimated_delivery_date, 1, 0) AS tardio
  FROM `olist-logistica.olist.orders`
  WHERE order_status = 'delivered'
    AND order_delivered_customer_date IS NOT NULL
)
"""

por_estado = sql(base + """
SELECT
  c.customer_state AS estado,
  COUNT(*) AS pedidos,
  ROUND(AVG(e.dias_real), 1) AS dias_real,
  ROUND(AVG(e.dias_prometidos), 1) AS dias_prometidos,
  ROUND(100 * AVG(e.tardio), 1) AS pct_tardios
FROM entregas e
JOIN `olist-logistica.olist.customers` c
  ON e.customer_id = c.customer_id
GROUP BY estado
HAVING COUNT(*) >= 100
ORDER BY pct_tardios DESC
""")

promesa_mes = sql(base + """
SELECT
  FORMAT_DATE('%Y-%m', DATE(order_purchase_timestamp)) AS mes,
  ROUND(AVG(dias_prometidos), 1) AS dias_prometidos,
  ROUND(AVG(dias_real), 1) AS dias_real,
  ROUND(100 * AVG(tardio), 1) AS pct_tardios
FROM entregas
WHERE order_purchase_timestamp >= '2017-01-01'
GROUP BY mes
ORDER BY mes
""")

display(por_estado)
display(promesa_mes)

,estado,pedidos,dias_real,dias_prometidos,pct_tardios
0,AL,397,24.5,32.6,23.9
1,MA,717,21.6,30.4,19.7
2,PI,476,19.4,30.1,16.0
3,CE,1279,21.2,31.4,15.3
4,SE,335,21.5,30.8,15.2
5,BA,3256,19.3,29.4,14.0
6,RJ,12350,15.3,26.3,13.5
7,TO,274,17.6,29.1,12.8
8,PA,946,23.8,37.1,12.4
9,ES,1995,15.8,25.6,12.2


,mes,dias_prometidos,dias_real,pct_tardios
0,2017-01,39.5,12.6,3.1
1,2017-02,31.8,13.1,3.2
2,2017-03,24.7,12.9,5.6
3,2017-04,27.3,14.9,7.9
4,2017-05,24.3,11.3,3.6
5,2017-06,24.0,12.0,3.9
6,2017-07,23.3,11.6,3.4
7,2017-08,23.5,11.1,3.3
8,2017-09,22.4,11.8,5.2
9,2017-10,23.0,11.8,5.3


In [7]:
def guardar(consulta, nombre):
    tabla = f"{PROYECTO}.olist.{nombre}"
    config = bigquery.QueryJobConfig(destination=tabla, write_disposition="WRITE_TRUNCATE")
    client.query(consulta, job_config=config).result()
    print(nombre, "->", f"{client.get_table(tabla).num_rows:,}", "filas")

guardar("""
SELECT
  geolocation_zip_code_prefix AS zip,
  AVG(geolocation_lat) AS lat,
  AVG(geolocation_lng) AS lng
FROM `olist-logistica.olist.geolocation`
GROUP BY zip
""", "geo_zip")

display(sql("SELECT * FROM `olist-logistica.olist.geo_zip` LIMIT 5"))

geo_zip -> 19,015 filas


,zip,lat,lng
0,69990,-7.605106,-72.916069
1,69983,-8.954550,-72.783659
2,69980,-7.626200,-72.668881
3,69985,-7.736559,-72.651635
4,69895,-7.546240,-72.583184


In [8]:
guardar("""
WITH items AS (
  SELECT
    i.order_id,
    COUNT(*) AS n_items,
    COUNT(DISTINCT i.seller_id) AS n_vendedores,
    SUM(i.price) AS precio_total,
    SUM(i.freight_value) AS flete_total,
    SUM(p.product_weight_g) AS peso_total_g,
    SUM(p.product_length_cm * p.product_height_cm * p.product_width_cm) AS volumen_cm3,
    MAX(i.shipping_limit_date) AS limite_vendedor,
    ANY_VALUE(s.seller_state) AS estado_vendedor,
    MAX(ST_DISTANCE(ST_GEOGPOINT(gs.lng, gs.lat), ST_GEOGPOINT(gc.lng, gc.lat))) / 1000 AS distancia_km
  FROM `olist-logistica.olist.order_items` i
  JOIN `olist-logistica.olist.orders` o ON i.order_id = o.order_id
  JOIN `olist-logistica.olist.customers` c ON o.customer_id = c.customer_id
  LEFT JOIN `olist-logistica.olist.products` p ON i.product_id = p.product_id
  LEFT JOIN `olist-logistica.olist.sellers` s ON i.seller_id = s.seller_id
  LEFT JOIN `olist-logistica.olist.geo_zip` gs ON s.seller_zip_code_prefix = gs.zip
  LEFT JOIN `olist-logistica.olist.geo_zip` gc ON c.customer_zip_code_prefix = gc.zip
  GROUP BY i.order_id
),
resenas AS (
  SELECT order_id, AVG(review_score) AS review_score
  FROM `olist-logistica.olist.order_reviews`
  GROUP BY order_id
)
SELECT
  o.order_id,
  o.order_purchase_timestamp AS fecha_compra,
  EXTRACT(DAYOFWEEK FROM o.order_purchase_timestamp) AS dia_semana,
  EXTRACT(HOUR FROM o.order_purchase_timestamp) AS hora,
  c.customer_state AS estado_cliente,
  it.estado_vendedor,
  ROUND(it.distancia_km, 1) AS distancia_km,
  it.n_items,
  it.n_vendedores,
  ROUND(it.precio_total, 2) AS precio_total,
  ROUND(it.flete_total, 2) AS flete_total,
  it.peso_total_g,
  it.volumen_cm3,
  ROUND(TIMESTAMP_DIFF(o.order_estimated_delivery_date, o.order_purchase_timestamp, HOUR) / 24, 1) AS dias_prometidos,
  ROUND(TIMESTAMP_DIFF(it.limite_vendedor, o.order_purchase_timestamp, HOUR) / 24, 1) AS dias_limite_vendedor,
  ROUND(TIMESTAMP_DIFF(o.order_delivered_customer_date,

_IncompleteInputError: incomplete input (2614715998.py, line 1)

In [9]:
guardar("""
SELECT
  i.order_id,
  COUNT(*) AS n_items,
  COUNT(DISTINCT i.seller_id) AS n_vendedores,
  SUM(i.price) AS precio_total,
  SUM(i.freight_value) AS flete_total,
  SUM(p.product_weight_g) AS peso_total_g,
  SUM(p.product_length_cm * p.product_height_cm * p.product_width_cm) AS volumen_cm3,
  MAX(i.shipping_limit_date) AS limite_vendedor,
  ANY_VALUE(s.seller_state) AS estado_vendedor,
  MAX(ST_DISTANCE(ST_GEOGPOINT(gs.lng, gs.lat), ST_GEOGPOINT(gc.lng, gc.lat))) / 1000 AS distancia_km
FROM `olist-logistica.olist.order_items` i
JOIN `olist-logistica.olist.orders` o ON i.order_id = o.order_id
JOIN `olist-logistica.olist.customers` c ON o.customer_id = c.customer_id
LEFT JOIN `olist-logistica.olist.products` p ON i.product_id = p.product_id
LEFT JOIN `olist-logistica.olist.sellers` s ON i.seller_id = s.seller_id
LEFT JOIN `olist-logistica.olist.geo_zip` gs ON s.seller_zip_code_prefix = gs.zip
LEFT JOIN `olist-logistica.olist.geo_zip` gc ON c.customer_zip_code_prefix = gc.zip
GROUP BY i.order_id
""", "items_pedido")

items_pedido -> 98,666 filas


In [10]:
guardar("""
SELECT order_id, AVG(review_score) AS review_score
FROM `olist-logistica.olist.order_reviews`
GROUP BY order_id
""", "resenas_pedido")

resenas_pedido -> 98,673 filas


In [11]:
guardar("""
SELECT
  o.order_id,
  o.order_purchase_timestamp AS fecha_compra,
  EXTRACT(DAYOFWEEK FROM o.order_purchase_timestamp) AS dia_semana,
  EXTRACT(HOUR FROM o.order_purchase_timestamp) AS hora,
  c.customer_state AS estado_cliente,
  it.estado_vendedor,
  ROUND(it.distancia_km, 1) AS distancia_km,
  it.n_items,
  it.n_vendedores,
  ROUND(it.precio_total, 2) AS precio_total,
  ROUND(it.flete_total, 2) AS flete_total,
  it.peso_total_g,
  it.volumen_cm3,
  ROUND(TIMESTAMP_DIFF(o.order_estimated_delivery_date, o.order_purchase_timestamp, HOUR) / 24, 1) AS dias_prometidos,
  ROUND(TIMESTAMP_DIFF(it.limite_vendedor, o.order_purchase_timestamp, HOUR) / 24, 1) AS dias_limite_vendedor,
  ROUND(TIMESTAMP_DIFF(o.order_delivered_customer_date, o.order_purchase_timestamp, HOUR) / 24, 1) AS dias_real,
  ROUND(TIMESTAMP_DIFF(o.order_delivered_customer_date, o.order_estimated_delivery_date, HOUR) / 24, 1) AS dias_retraso,
  IF(o.order_delivered_customer_date > o.order_estimated_delivery_date, 1, 0) AS tardio,
  r.review_score
FROM `olist-logistica.olist.orders` o
JOIN `olist-logistica.olist.customers` c ON o.customer_id = c.customer_id
JOIN `olist-logistica.olist.items_pedido` it ON o.order_id = it.order_id
LEFT JOIN `olist-logistica.olist.resenas_pedido` r ON o.order_id = r.order_id
WHERE o.order_status = 'delivered'
  AND o.order_delivered_customer_date IS NOT NULL
""", "pedidos_modelo")

pedidos_modelo -> 96,470 filas


In [12]:
sql("""
SELECT
  COUNT(*) AS pedidos,
  COUNTIF(distancia_km IS NULL) AS sin_distancia,
  COUNTIF(peso_total_g IS NULL) AS sin_peso,
  COUNTIF(review_score IS NULL) AS sin_resena,
  ROUND(AVG(distancia_km)) AS distancia_media_km,
  ROUND(100 * AVG(tardio), 1) AS pct_tardios
FROM `olist-logistica.olist.pedidos_modelo`
""")

,pedidos,sin_distancia,sin_peso,sin_resena,distancia_media_km,pct_tardios
0,96470,476,16,646,602.0,8.1


In [13]:
import pandas as pd
import numpy as np

df = sql("SELECT * FROM `olist-logistica.olist.pedidos_modelo`")
print(df.shape)

# 1) Impacto del retraso en la calificación
df["rango_retraso"] = pd.cut(
    df["dias_retraso"],
    bins=[-999, 0, 3, 7, 999],
    labels=["A tiempo", "1-3 días tarde", "4-7 días tarde", "Más de 7 días tarde"]
)

impacto = df.groupby("rango_retraso", observed=True).agg(
    pedidos=("order_id", "count"),
    calificacion_promedio=("review_score", "mean"),
    pct_calificacion_baja=("review_score", lambda s: 100 * (s.dropna() <= 2).mean()),
).round(2)
print(impacto)
print()

# 2) Retrasos según la distancia
df["rango_km"] = pd.cut(df["distancia_km"], bins=[0, 100, 300, 600, 1000, 2000, 5000])

por_distancia = df.groupby("rango_km", observed=True).agg(
    pedidos=("order_id", "count"),
    dias_real=("dias_real", "mean"),
    pct_tardios=("tardio", lambda s: 100 * s.mean()),
).round(1)
print(por_distancia)

(96470, 19)
                     pedidos  calificacion_promedio  pct_calificacion_baja
rango_retraso                                                             
A tiempo               88722                   4.29                   9.18
1-3 días tarde          2610                   3.74                  19.82
4-7 días tarde          1829                   2.29                  61.83
Más de 7 días tarde     3309                   1.73                  78.30

              pedidos  dias_real  pct_tardios
rango_km                                     
(0, 100]        17783        6.5          6.4
(100, 300]      13307       10.0          6.5
(300, 600]      31515       12.6          7.7
(600, 1000]     17965       14.7          8.4
(1000, 2000]     9777       17.9         10.8
(2000, 5000]     5621       21.1         13.6


In [14]:
 # Solo desde 2017 (2016 tiene muy pocos pedidos)
datos = df[df["fecha_compra"] >= pd.Timestamp("2017-01-01", tz="UTC")].copy()
datos["mes"] = datos["fecha_compra"].dt.month

numericas = ["dia_semana", "hora", "mes", "distancia_km", "n_items", "n_vendedores",
             "precio_total", "flete_total", "peso_total_g", "volumen_cm3",
             "dias_prometidos", "dias_limite_vendedor"]
categoricas = ["estado_cliente", "estado_vendedor"]

datos[numericas] = datos[numericas].astype("float64")
for c in categoricas:
    datos[c] = datos[c].astype("category")

X = datos[numericas + categoricas]
y = datos["tardio"].astype(int)

# Separación por fecha: entrenar con el pasado, probar con el futuro
fin_train = pd.Timestamp("2018-04-15", tz="UTC")
inicio_test = pd.Timestamp("2018-05-01", tz="UTC")

train_mask = datos["fecha_compra"] < fin_train
test_mask = datos["fecha_compra"] >= inicio_test

X_train, y_train = X[train_mask], y[train_mask]
X_test, y_test = X[test_mask], y[test_mask]

print("Entrenamiento:", X_train.shape, "| % tardíos:", round(100 * y_train.mean(), 1))
print("Prueba:       ", X_test.shape, "| % tardíos:", round(100 * y_test.mean(), 1))

Entrenamiento: (67112, 14) | % tardíos: 9.1
Prueba:        (25352, 14) | % tardíos: 6.2


In [15]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

modelo = HistGradientBoostingClassifier(categorical_features="from_dtype", random_state=42)
modelo.fit(X_train, y_train)
prob = modelo.predict_proba(X_test)[:, 1]

def resumen(nombre, puntaje):
    roc = roc_auc_score(y_test, puntaje)
    pr = average_precision_score(y_test, puntaje)
    print(f"{nombre:30s} ROC-AUC: {roc:.3f}   PR-AUC: {pr:.3f}")

print("Tasa base de tardíos (referencia de PR-AUC):", round(y_test.mean(), 3))
print("Exactitud de 'nadie se retrasa':", round(1 - y_test.mean(), 3))
print()
resumen("Regla: más distancia", X_test["distancia_km"].fillna(X_test["distancia_km"].median()))
resumen("Regla: menor plazo prometido", -X_test["dias_prometidos"])
resumen("Gradient Boosting", prob)

Tasa base de tardíos (referencia de PR-AUC): 0.062
Exactitud de 'nadie se retrasa': 0.938

Regla: más distancia           ROC-AUC: 0.438   PR-AUC: 0.061
Regla: menor plazo prometido   ROC-AUC: 0.715   PR-AUC: 0.198
Gradient Boosting              ROC-AUC: 0.736   PR-AUC: 0.136


In [16]:
# Tiempo típico por ruta (estado vendedor -> estado cliente), calculado SOLO con entrenamiento
tmp = datos.loc[train_mask, ["estado_vendedor", "estado_cliente", "dias_real"]].copy()
tmp["estado_vendedor"] = tmp["estado_vendedor"].astype(str)
tmp["estado_cliente"] = tmp["estado_cliente"].astype(str)
tmp["dias_real"] = tmp["dias_real"].astype(float)

ruta = tmp.groupby(["estado_vendedor", "estado_cliente"])["dias_real"].median().to_dict()
mediana_global = tmp["dias_real"].median()

def tiempo_tipico(X_):
    pares = zip(X_["estado_vendedor"].astype(str), X_["estado_cliente"].astype(str))
    return np.array([ruta.get(p, mediana_global) for p in pares])

X_train2 = X_train.copy()
X_test2 = X_test.copy()
for A in (X_train2, X_test2):
    A["dias_ruta_tipico"] = tiempo_tipico(A)
    A["colchon"] = A["dias_prometidos"] - A["dias_ruta_tipico"]

modelo2 = HistGradientBoostingClassifier(categorical_features="from_dtype", random_state=42)
modelo2.fit(X_train2, y_train)
prob2 = modelo2.predict_proba(X_test2)[:, 1]

print("Tasa base:", round(y_test.mean(), 3))
resumen("Regla: menor colchón", -X_test2["colchon"])
resumen("Gradient Boosting (v1)", prob)
resumen("Gradient Boosting + colchón (v2)", prob2)

Tasa base: 0.062
Regla: menor colchón           ROC-AUC: 0.756   PR-AUC: 0.190
Gradient Boosting (v1)         ROC-AUC: 0.736   PR-AUC: 0.136
Gradient Boosting + colchón (v2) ROC-AUC: 0.757   PR-AUC: 0.157


In [17]:
def entrenar(columnas, **parametros):
    m = HistGradientBoostingClassifier(categorical_features="from_dtype", random_state=42, **parametros)
    m.fit(X_train2[columnas], y_train)
    return m.predict_proba(X_test2[columnas])[:, 1]

todas = list(X_train2.columns)
sin_mes = [c for c in todas if c != "mes"]
sin_mes_ni_estados = [c for c in sin_mes if c not in categoricas]

regularizado = dict(max_depth=3, learning_rate=0.05, max_iter=200,
                    min_samples_leaf=200, l2_regularization=5.0)

print("Tasa base:", round(y_test.mean(), 3))
resumen("v2 completo (14+2 variables)", prob2)
resumen("sin mes", entrenar(sin_mes))
resumen("sin mes ni estados", entrenar(sin_mes_ni_estados))
resumen("sin mes ni estados + regulariz.", entrenar(sin_mes_ni_estados, **regularizado))
resumen("Regla: menor colchón", -X_test2["colchon"])

Tasa base: 0.062
v2 completo (14+2 variables)   ROC-AUC: 0.757   PR-AUC: 0.157
sin mes                        ROC-AUC: 0.705   PR-AUC: 0.120
sin mes ni estados             ROC-AUC: 0.705   PR-AUC: 0.116
sin mes ni estados + regulariz. ROC-AUC: 0.675   PR-AUC: 0.108
Regla: menor colchón           ROC-AUC: 0.756   PR-AUC: 0.190


In [18]:
# Fechas como arreglos de numpy (en UTC, sin zona horaria)
t = datos["fecha_compra"].dt.tz_localize(None).to_numpy()
dias_ns = (datos["dias_real"].astype(float).to_numpy() * 86400 * 1e9).astype("int64")
entrega = t + dias_ns.astype("timedelta64[ns]")          # momento en que llegó cada pedido
tardio_arr = datos["tardio"].astype(int).to_numpy()

# Pedidos ordenados por fecha de entrega, con acumulados de tardíos
orden = np.argsort(entrega)
entrega_ord = entrega[orden]
acum_tardios = np.concatenate([[0], np.cumsum(tardio_arr[orden])])

def tasa_tardios_reciente(dias, minimo=30):
    hasta = np.searchsorted(entrega_ord, t, side="right")
    desde = np.searchsorted(entrega_ord, t - np.timedelta64(dias, "D"), side="right")
    n = hasta - desde
    tardios = acum_tardios[hasta] - acum_tardios[desde]
    return np.where(n >= minimo, tardios / np.maximum(n, 1), np.nan)

# Volumen: pedidos recibidos en los 7 días previos
t_ord = np.sort(t)
pedidos_7d = (np.searchsorted(t_ord, t, side="right")
              - np.searchsorted(t_ord, t - np.timedelta64(7, "D"), side="right"))

datos["tasa_tardios_14d"] = tasa_tardios_reciente(14)
datos["pedidos_7d"] = pedidos_7d

X_train3 = X_train2.copy()
X_test3 = X_test2.copy()
for nombre in ["tasa_tardios_14d", "pedidos_7d"]:
    X_train3[nombre] = datos.loc[X_train3.index, nombre].astype(float)
    X_test3[nombre] = datos.loc[X_test3.index, nombre].astype(float)

def entrenar3(columnas):
    m = HistGradientBoostingClassifier(categorical_features="from_dtype", random_state=42)
    m.fit(X_train3[columnas], y_train)
    return m.predict_proba(X_test3[columnas])[:, 1]

cols3 = list(X_train3.columns)
print("Tasa base:", round(y_test.mean(), 3))
resumen("v2 (sin contexto)", prob2)
resumen("v3 con contexto", entrenar3(cols3))
resumen("v3 con contexto, sin mes", entrenar3([c for c in cols3 if c != "mes"]))
resumen("Regla: menor colchón", -X_test2["colchon"])

Tasa base: 0.062
v2 (sin contexto)              ROC-AUC: 0.757   PR-AUC: 0.157
v3 con contexto                ROC-AUC: 0.659   PR-AUC: 0.099
v3 con contexto, sin mes       ROC-AUC: 0.676   PR-AUC: 0.106
Regla: menor colchón           ROC-AUC: 0.756   PR-AUC: 0.190


In [19]:
filas = []
for col in X_train3.columns:
    if str(X_train3[col].dtype) == "category":
        continue
    a = X_train3[col].astype(float)
    b = X_test3[col].astype(float)
    mediana = a.median()
    auc_train = roc_auc_score(y_train, a.fillna(mediana))
    auc_test = roc_auc_score(y_test, b.fillna(mediana))
    filas.append((col, round(auc_train, 3), round(auc_test, 3)))

tabla = pd.DataFrame(filas, columns=["variable", "AUC_train", "AUC_test"])
tabla["diferencia"] = (tabla["AUC_test"] - tabla["AUC_train"]).round(3)
print(tabla.sort_values("diferencia").to_string(index=False))

            variable  AUC_train  AUC_test  diferencia
     dias_prometidos      0.481     0.285      -0.196
    dias_ruta_tipico      0.631     0.458      -0.173
        distancia_km      0.587     0.438      -0.149
         flete_total      0.583     0.442      -0.141
    tasa_tardios_14d      0.587     0.462      -0.125
             colchon      0.367     0.244      -0.123
dias_limite_vendedor      0.490     0.441      -0.049
          dia_semana      0.497     0.471      -0.026
        precio_total      0.528     0.503      -0.025
         volumen_cm3      0.510     0.492      -0.018
        peso_total_g      0.517     0.506      -0.011
        n_vendedores      0.495     0.492      -0.003
             n_items      0.490     0.491       0.001
                hora      0.510     0.515       0.005
          pedidos_7d      0.669     0.685       0.016
                 mes      0.469     0.547       0.078


In [20]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

def construir(mascara_train, mascara_test):
    tmp = datos.loc[mascara_train, ["estado_vendedor", "estado_cliente", "dias_real"]].copy()
    tmp["estado_vendedor"] = tmp["estado_vendedor"].astype(str)
    tmp["estado_cliente"] = tmp["estado_cliente"].astype(str)
    tmp["dias_real"] = tmp["dias_real"].astype(float)
    ruta = tmp.groupby(["estado_vendedor", "estado_cliente"])["dias_real"].median().to_dict()
    mediana_global = tmp["dias_real"].median()

    salida = []
    for mascara in (mascara_train, mascara_test):
        A = X[mascara].copy()
        pares = zip(A["estado_vendedor"].astype(str), A["estado_cliente"].astype(str))
        A["dias_ruta_tipico"] = np.array([ruta.get(p, mediana_global) for p in pares])
        A["colchon"] = A["dias_prometidos"] - A["dias_ruta_tipico"]
        salida.append(A)
    return salida

columnas_completo = numericas + categoricas + ["dias_ruta_tipico", "colchon"]
columnas_estable = ["colchon", "dias_prometidos", "dias_limite_vendedor", "dia_semana", "hora",
                    "precio_total", "peso_total_g", "volumen_cm3", "n_items", "n_vendedores", "mes"]

def gb(columnas, Xa, ya, Xb):
    m = HistGradientBoostingClassifier(categorical_features="from_dtype", random_state=42)
    m.fit(Xa[columnas], ya)
    return m.predict_proba(Xb[columnas])[:, 1]

meses = pd.date_range("2017-10-01", "2018-08-01", freq="MS", tz="UTC")
filas = []

for inicio in meses:
    fin = inicio + pd.offsets.MonthBegin(1)
    m_train = datos["fecha_compra"] < inicio - pd.Timedelta(days=15)
    m_test = (datos["fecha_compra"] >= inicio) & (datos["fecha_compra"] < fin)

    Xa, Xb = construir(m_train, m_test)
    ya, yb = y[m_train], y[m_test]
    if yb.sum() < 50:
        continue

    lr = make_pipeline(StandardScaler(), LogisticRegression())
    lr.fit(Xa[["colchon", "dias_prometidos"]], ya)

    puntajes = {
        "Regla colchón": -Xb["colchon"],
        "Logística (2 var.)": lr.predict_proba(Xb[["colchon", "dias_prometidos"]])[:, 1],
        "GB completo": gb(columnas_completo, Xa, ya, Xb),
        "GB estable": gb(columnas_estable, Xa, ya, Xb),
    }
    for nombre, p in puntajes.items():
        filas.append({"ventana": inicio.strftime("%Y-%m"), "modelo": nombre,
                      "tasa_base": yb.mean(),
                      "pr_auc": average_precision_score(yb, p),
                      "roc_auc": roc_auc_score(yb, p)})

res_cls = pd.DataFrame(filas)
print(res_cls.pivot(index="ventana", columns="modelo", values="pr_auc").round(3))
print()
print(res_cls.groupby("modelo")[["pr_auc", "roc_auc"]].mean().round(3))
print("Tasa base promedio:", round(res_cls.drop_duplicates("ventana")["tasa_base"].mean(), 3))

modelo   GB completo  GB estable  Logística (2 var.)  Regla colchón
ventana                                                            
2017-10        0.101       0.095               0.101          0.090
2017-11        0.198       0.186               0.209          0.152
2017-12        0.125       0.103               0.123          0.085
2018-01        0.099       0.105               0.121          0.081
2018-02        0.276       0.244               0.298          0.208
2018-03        0.353       0.309               0.362          0.237
2018-04        0.187       0.187               0.210          0.108
2018-05        0.170       0.179               0.207          0.151
2018-06        0.072       0.077               0.058          0.035
2018-07        0.144       0.113               0.104          0.148
2018-08        0.165       0.188               0.136          0.281

                    pr_auc  roc_auc
modelo                             
GB completo          0.172    0.681
GB esta

In [21]:
filas = []
for inicio in meses:
    fin = inicio + pd.offsets.MonthBegin(1)
    m_train = datos["fecha_compra"] < inicio - pd.Timedelta(days=15)
    m_test = (datos["fecha_compra"] >= inicio) & (datos["fecha_compra"] < fin)

    Xa, Xb = construir(m_train, m_test)
    ya, yb = y[m_train], y[m_test]
    if yb.sum() < 50:
        continue

    lr = make_pipeline(StandardScaler(), LogisticRegression())
    lr.fit(Xa[["colchon", "dias_prometidos"]], ya)

    puntajes = {
        "Logística (2 var.)": lr.predict_proba(Xb[["colchon", "dias_prometidos"]])[:, 1],
        "GB completo": gb(columnas_completo, Xa, ya, Xb),
    }
    for nombre, p in puntajes.items():
        orden = np.argsort(-np.asarray(p))
        reales = yb.to_numpy()
        for frac in (0.10, 0.20):
            k = int(len(yb) * frac)
            capturados = reales[orden[:k]].sum()
            filas.append({"ventana": inicio.strftime("%Y-%m"), "modelo": nombre,
                          "revisar": f"{int(frac * 100)}%",
                          "retrasos_capturados": 100 * capturados / reales.sum()})

top = pd.DataFrame(filas)
print("% de retrasos reales capturados al revisar el 10% y el 20% de pedidos más riesgosos:")
print(top.groupby(["modelo", "revisar"])["retrasos_capturados"].mean().round(1).unstack())

# ¿Qué dice la regresión logística, en palabras?
m_final = datos["fecha_compra"] < pd.Timestamp("2018-07-15", tz="UTC")
Xf, _ = construir(m_final, m_final)
lr_final = LogisticRegression(max_iter=1000).fit(Xf[["colchon", "dias_prometidos"]], y[m_final])
efecto = pd.Series(np.exp(lr_final.coef_[0]), index=["colchon", "dias_prometidos"]).round(3)
print()
print(efecto)
print("Un día más de plazo prometido (misma ruta): las probabilidades de retraso se multiplican por", round(efecto.prod(), 3))
print("Una ruta que tarda 1 día más (mismo plazo): se multiplican por", round(1 / efecto["colchon"], 3))

% de retrasos reales capturados al revisar el 10% y el 20% de pedidos más riesgosos:
revisar              10%   20%
modelo                        
GB completo         24.5  40.6
Logística (2 var.)  24.8  41.6

colchon            0.810
dias_prometidos    1.119
dtype: float64
Un día más de plazo prometido (misma ruta): las probabilidades de retraso se multiplican por 0.906
Una ruta que tarda 1 día más (mismo plazo): se multiplican por 1.235
